In [32]:
import networkx as nx
from torch_geometric.datasets import TUDataset

# Specify the name of the dataset you want to load
dataset_name = 'ENZYMES'  # Example dataset name

# Load the dataset
dataset = TUDataset(root='data/TUDataset', name=dataset_name)


In [33]:
# Get the first graph
data = dataset[0]

# Access node features (shape: [num_nodes, num_features])
node_features = data.x

# Access node labels (shape: [num_nodes])
node_labels = data.y

In [34]:
import torch

This code sets up a basic neural network model called a Multi-Layer Perceptron (MLP) using PyTorch. The model has one hidden layer and uses ReLU and log-softmax functions to process inputs and make predictions. The model is created with specific input, hidden, and output sizes based on the dataset. 

In [35]:
import torch.nn.functional as F

class MLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(MLP, self).__init__()
        self.lin1 = torch.nn.Linear(in_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x):
        x = F.relu(self.lin1(x))
        x = F.log_softmax(self.lin2(x), dim=1)
        return x

# Define model parameters
in_channels = dataset.num_node_features
hidden_channels = 64
out_channels = dataset.num_classes  # Number of enzyme classes (6)

# Create the model instance
model = MLP(in_channels, hidden_channels, out_channels)

In [36]:
data.y.shape

torch.Size([1])

In [37]:
data.x.shape

torch.Size([37, 3])

In [38]:
node_features = data.x
node_labels = data.y

This creates a data transformation for splitting graph data into training, validation, and test sets using specified ratios. It then loads a dataset from the TUDataset, applying the transformation during loading. Finally, it splits the dataset into training, validation, and test sets and prints them.

In [39]:
import torch_geometric.transforms as T

In [40]:
transform = T.Compose([
    T.RandomLinkSplit(num_val=0.05,  # ratio of edges including in the validation set
                      num_test=0.2,  # ratio of edges including in the test set
                      is_undirected=True,
                      add_negative_train_samples=False),
])

# dataset = SNAPDataset('/tmp/snap', 'ego-facebook', transform=transform)
dataset = TUDataset(root='data/TUDataset', name=dataset_name, transform=transform)

train_data, val_data, test_data = dataset[0]
print(train_data, val_data, test_data, sep="\n")


Data(edge_index=[2, 128], x=[37, 3], y=[1], edge_label=[64], edge_label_index=[2, 64])
Data(edge_index=[2, 128], x=[37, 3], y=[1], edge_label=[8], edge_label_index=[2, 8])
Data(edge_index=[2, 136], x=[37, 3], y=[1], edge_label=[32], edge_label_index=[2, 32])


In [41]:
# dataset = SNAPDataset('/tmp/snap', 'ego-facebook', transform=transform)
dataset = TUDataset(root='data/TUDataset', name=dataset_name, transform=transform)

In [42]:
print("Number of the nodes in training, validation and test data are", train_data.num_nodes, val_data.num_nodes, test_data.num_nodes)
print("Number of the edges in training, validation and test data are", train_data.num_edges, val_data.num_edges, test_data.num_edges)
print("Number of the edge_label_index in training, validation and test data are", train_data.edge_label_index.shape[1], 
                                                                                  val_data.edge_label_index.shape[1],
                                                                                  test_data.edge_label_index.shape[1])

Number of the nodes in training, validation and test data are 37 37 37
Number of the edges in training, validation and test data are 128 128 136
Number of the edge_label_index in training, validation and test data are 64 8 32


This defines a Graph Convolutional Network (GCN) model in PyTorch with two graph convolutional layers and a ReLU activation function. The model processes input node features and edge indices to produce output node features. The forward method applies the first convolutional layer and ReLU activation, followed by the second convolutional layer.

In [43]:
from torch_geometric.nn import GCNConv
import torch
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.act = torch.nn.ReLU()

    def forward(self, node_feature, edge_index):

        output = self.conv1(node_feature, edge_index)
        output = self.act(output)
        output = self.conv2(output, edge_index)

        return output

In [44]:
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)

In [45]:
model = GCN(dataset.num_features, hidden_channels=128, out_channels=64)

This defines a function to compute the similarity between node embeddings for given pairs of nodes. The function normalizes the node embeddings and calculates the inner product for each pair of nodes specified in the edge index. Finally, it prints the computed similarity values for the example node embeddings and edge index.

In [58]:
def compute_similarity(node_embs, edge_index):
    result = torch.tensor([0.] * len(edge_index[0]))

    node_embs = torch.nn.functional.normalize(node_embs)
    for i in range(len(edge_index[0])):
      result[i] = torch.dot(node_embs[edge_index[0][i]], node_embs[edge_index[1][i]])


    return result

n, h = 5, 10  # number of nodes and embedding size
node_embs = torch.rand(n, h)
edge_index = torch.tensor([[0, 1, 2, 3], 
                           [2, 3, 0, 1]])  # compute the similarity of (0, 2), (1, 3), (2, 0), (3, 1)
similarity = compute_similarity(node_embs, edge_index)
print("Similairty:", similarity)

Similairty: tensor([0.7518, 0.7928, 0.7518, 0.7928])


This code defines a training function for a graph neural network model. It performs forward propagation, negative sampling to generate negative edges, and computes the loss using a specified loss function. The loss is then backpropagated, and the optimizer updates the model parameters.

In [70]:
def train(model, data, optimizer, loss_fn):

    loss = 0

    
    model.train()
    optimizer.zero_grad()
    y_pred = model(data.x, data.edge_index)
    
    neg_edge_index = negative_sampling(
      edge_index=train_data.edge_index,  # positive edges in the graph
      num_nodes=train_data.num_nodes,  # number of nodes
      num_neg_samples=len(data.edge_label),  # number of negative examples
    )
    # print(neg_edge_index.shape)
    train_edge_index = torch.cat((data.edge_label_index, neg_edge_index), dim=1)
    train_edge_label = torch.cat((data.edge_label, torch.tensor([0] * len(train_data.edge_label_index[0]))))
    # print(train_edge_index.shape)
    pred = compute_similarity(data.x, train_edge_index)
    loss = loss_fn(pred.clone().detach().requires_grad_(True), train_edge_label)
    loss.backward()
    optimizer.step()


    return loss

This function evaluates a graph-based machine learning model using the ROC AUC score from scikit-learn's metrics module. It operates within a `torch.no_grad()` context to disable gradient tracking during inference, ensuring efficiency. The model computes similarities between nodes using message passing on graph edges defined by `edge_index`.


In [71]:
from sklearn.metrics import roc_auc_score

@torch.no_grad()
def test(model, data):
    model.eval()
    out = model(data.x, data.edge_index)  # use `edge_index` to perform message passing
    out = compute_similarity(out, data.edge_label_index).view(-1).sigmoid()  # use `edge_label_index` to compute the loss
    return roc_auc_score(data.edge_label.cpu().numpy(), out.cpu().numpy())

In [72]:
loss_fn = torch.nn.BCEWithLogitsLoss()

In [73]:
from torch_geometric.utils import negative_sampling

neg_edge_index = negative_sampling(
      edge_index=train_data.edge_index,  # positive edges in the graph
      num_nodes=train_data.num_nodes,  # number of nodes
      num_neg_samples=5,  # number of negative examples
    )

print("shape of neg_edge_index:", neg_edge_index.shape)  # [2, num_neg_samples]
print("negative examples:", neg_edge_index)

shape of neg_edge_index: torch.Size([2, 5])
negative examples: tensor([[ 3, 33,  1, 12, 35],
        [19, 10, 28, 35,  8]])


This loop trains a machine learning model over 50 epochs, optimizing using a specified loss function and updating parameters with an optimizer. It evaluates the model's performance on validation and test datasets using the ROC AUC score, storing the best validation score and corresponding test score for final evaluation.


In [74]:
epochs = 50

best_val_auc = final_test_auc = 0
for epoch in range(1, epochs + 1):
    loss = train(model, train_data, optimizer, loss_fn)
    valid_auc = test(model, val_data)
    test_auc = test(model, test_data)
    if valid_auc > best_val_auc:
        best_val_auc = valid_auc
        final_test_auc = test_auc
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Val: {valid_auc:.4f}, Test: {test_auc:.4f}')

Epoch: 001, Loss: 0.6745, Val: 0.6250, Test: 0.4883
Epoch: 002, Loss: 0.7132, Val: 0.6250, Test: 0.4883
Epoch: 003, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 004, Loss: 0.7084, Val: 0.6250, Test: 0.4883
Epoch: 005, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 006, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 007, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 008, Loss: 0.6648, Val: 0.6250, Test: 0.4883
Epoch: 009, Loss: 0.6551, Val: 0.6250, Test: 0.4883
Epoch: 010, Loss: 0.7035, Val: 0.6250, Test: 0.4883
Epoch: 011, Loss: 0.7035, Val: 0.6250, Test: 0.4883
Epoch: 012, Loss: 0.7181, Val: 0.6250, Test: 0.4883
Epoch: 013, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 014, Loss: 0.6551, Val: 0.6250, Test: 0.4883
Epoch: 015, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 016, Loss: 0.6938, Val: 0.6250, Test: 0.4883
Epoch: 017, Loss: 0.7132, Val: 0.6250, Test: 0.4883
Epoch: 018, Loss: 0.6987, Val: 0.6250, Test: 0.4883
Epoch: 019, Loss: 0.7132, Val: 0.6250, Test: 0.4883
Epoch: 020, 

In [75]:
print(f'The accuracy score is: {final_test_acc:.4f}')

The accuracy score is: 0.4883
